# CE541E08 — Unit 3 · Day 19 — Introduction to NumPy

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 19 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | Why NumPy · ndarray · statistics · visualising rainfall |

---
> **How to use this notebook:**
> Read the explanation in each section first. Trace through the algorithm to understand the logic. Then run the code cell and verify the output matches your expectation.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 19"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Why NumPy?

### What is the problem with Python lists?

Suppose you have 30 days of rainfall data stored in a list. If you want to add 5 mm to every value (to correct a sensor offset), you need a `for` loop that goes through every element one by one. For 30 values this is manageable. For 30 years of daily data — that is 10,950 values — a loop becomes slow and the code becomes cluttered.

**NumPy solves this.** NumPy stores all values in a compact block of memory and uses highly optimised C code internally. When you write `rainfall + 5`, NumPy adds 5 to every element simultaneously. This is called **vectorisation** — the operation is applied to the whole array as a vector, not element by element.

### Why does this matter for Civil Engineering?

In hydrology and hydraulics, datasets are large:
- A single rain gauge records 365 values per year
- A basin may have 50 gauges
- Analysis often spans 30+ years of records

Without NumPy, processing such data in Python would be impractically slow. With NumPy, operations that would need thousands of loop iterations happen in one line.

### Key comparison

| Task | Python list approach | NumPy approach |
|---|---|---|
| Add 5 to every rainfall value | `for` loop — 4 lines | `rainfall + 5` — 1 line |
| Find all values above 64.5 mm | `for` loop + `if` | `rainfall[rainfall > 64.5]` — 1 line |
| Compute the mean | `sum(lst) / len(lst)` | `rainfall.mean()` |
| Speed (1 million values) | Slow (several seconds) | ~100× faster |

### The ndarray

The core NumPy object is the **ndarray** (n-dimensional array):
- Every element is the **same data type** (e.g. all `float64`), unlike a Python list which can mix types
- It has a fixed **shape** — for a list of 30 values, shape is `(30,)`
- Mathematical operations apply to the entire array at once

---
## Code Block 1 — Creating an Array and Computing Statistics

### What this code does

We store 30 days of daily rainfall for the Cauvery basin (June 2024) in a NumPy array, then compute six statistics — total, mean, maximum, data type, shape, and minimum of rainy days — all without writing a single loop.

### Why each step is taken

**Import NumPy as `np`:**
Every NumPy program starts with this line. The alias `np` is the universal convention — you will see it in every textbook, paper, and tutorial worldwide. Writing `np.` instead of `numpy.` saves typing and is immediately recognisable.

**Create the array with `np.array([...])`:**
We pass a Python list inside `np.array()`. NumPy reads the list, allocates a contiguous block of memory, and stores all values as 64-bit floating point numbers (`float64`). From this point on, every operation on `rainfall` is vectorised — it applies to all 30 values simultaneously.

**Check `dtype` and `shape` before computing:**
Before computing anything, it is good practice to confirm the array is what you expect. `dtype` tells you the data type. `shape` tells you the dimensions — `(30,)` means a 1-D array of 30 elements. Catching a wrong shape early prevents incorrect results later.

**Use `.sum()`, `.mean()`, `.max()`:**
These are methods of the ndarray. They operate on the entire array in one call, with no loop and no intermediate variable needed.

**Boolean indexing `rainfall[rainfall > 0]`:**
`rainfall > 0` does not return a single True or False — it returns an array of 30 True/False values, one per element. Using that as an index (`rainfall[...]`) selects only the elements where the condition is True. We then call `.min()` on that filtered array to find the smallest rainy-day value. This would take a for loop and an if statement in plain Python — here it is one expression.

### Algorithm

```
1. Import numpy as np

2. Create rainfall array using np.array()
   — converts Python list to ndarray stored in contiguous memory

3. Print dtype (data type) and shape (dimensions)
   — confirms the array is correct before any computation

4. Compute statistics using array methods:
   rainfall.sum()   → total rainfall for the month
   rainfall.mean()  → average daily rainfall
   rainfall.max()   → highest single-day rainfall

5. Apply boolean filter:
   rainfall > 0  →  produces True/False for each of the 30 elements
   rainfall[rainfall > 0]  →  keeps only the elements where True
   .min()  →  finds the smallest value in the filtered result
   = smallest rainfall on any rainy day

6. Print all results with labels and units
```

In [ ]:
import numpy as np

# np.array() converts a Python list into a NumPy ndarray
# All 30 values are stored as float64 (64-bit decimal numbers)
rainfall = np.array([0, 0, 12.4, 45.6, 0, 8.2, 23.1,
                     0, 0, 87.3, 34.5, 0, 56.2, 0,
                     18.9, 0, 0, 134.5, 22.3, 45.6,
                     0, 0, 67.8, 12.1, 0, 89.4, 33.2,
                     0, 45.1, 28.7])

# Check array properties — always do this first
# dtype tells us how values are stored
# shape tells us dimensions: (30,) means 1-D with 30 elements
print("dtype  :", rainfall.dtype)
print("shape  :", rainfall.shape)
print("size   :", rainfall.size, "values")

# Statistics — each is a single method call on the entire array
# No loop needed; NumPy handles all 30 values internally
print()
print("total  :", rainfall.sum(), "mm")
print("mean   :", round(rainfall.mean(), 2), "mm/day")
print("max    :", rainfall.max(), "mm")

# Boolean indexing:
# rainfall > 0  →  array of True/False: [F, F, T, T, F, T, T, ...]
# rainfall[rainfall > 0]  →  selects only the True elements: [12.4, 45.6, 8.2, ...]
# .min()  →  finds the smallest value in that filtered selection
print("min>0  :", rainfall[rainfall > 0].min(), "mm")

---
## Code Block 2 — Visualising Rainfall as a Bar Chart

### What this code does

We plot the 30-day rainfall dataset as a bar chart. Two horizontal reference lines are added — one for the IMD Heavy Rain threshold (64.5 mm) and one for the monthly mean. This immediately makes it visible which days triggered a dam safety alert.

### Why each step is taken

**`np.arange(1, 31)` for the x-axis:**
We need day numbers 1 to 30 on the x-axis. `np.arange(1, 31)` generates `[1, 2, 3, ..., 30]` automatically. This is more reliable than writing all 30 numbers by hand and avoids off-by-one errors.

**`plt.figure(figsize=(10, 4))`:**
This sets the canvas size in inches (width × height). A wide, short figure suits a time series — day numbers stay readable without overlapping. Without this, Matplotlib uses a default size that may be too small.

**`plt.bar(days, rainfall)`:**
A bar chart is the correct choice for daily rainfall because each day is a discrete event — there is no meaningful value between Day 3 and Day 4. A line chart would imply continuity between dry days and rainy days, which is misleading.

**`plt.axhline(64.5, ...)`:**
`axhline` draws a horizontal line across the full width of the plot at a fixed y-value. The 64.5 mm line is the IMD threshold for Heavy Rain. Any bar taller than this line represents a day when dam safety procedures were triggered. The line makes this threshold immediately visible without any mental calculation.

**`plt.axhline(rainfall.mean(), ...)`:**
The mean line gives context. It shows what a typical day looks like and separates above-average from below-average days at a glance.

**`(rainfall >= 64.5).sum()`:**
This counts Heavy+ days using the same boolean indexing logic from Code Block 1. `>=` produces a True/False array and `.sum()` counts the True values (True = 1, False = 0 in NumPy).

### Algorithm

```
1. Create day numbers: np.arange(1, 31) → [1, 2, 3, ..., 30]

2. Open figure canvas: 10 inches wide, 4 inches tall

3. Draw bar chart: x = day numbers, y = rainfall (mm)

4. Draw orange dashed horizontal line at y = 64.5 mm
   — marks the IMD Heavy Rain threshold
   — any bar above this line = dam safety alert day

5. Draw red dotted horizontal line at y = rainfall.mean()
   — shows the monthly average for comparison

6. Add axis labels, title, and legend
   — plt.tight_layout() prevents labels being cut off

7. plt.show() — render and display the chart

8. Count Heavy+ days:
   rainfall >= 64.5  →  True/False array
   .sum()  →  counts True values (True=1, False=0)
   Print the count
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rainfall = np.array([0, 0, 12.4, 45.6, 0, 8.2, 23.1,
                     0, 0, 87.3, 34.5, 0, 56.2, 0,
                     18.9, 0, 0, 134.5, 22.3, 45.6,
                     0, 0, 67.8, 12.1, 0, 89.4, 33.2,
                     0, 45.1, 28.7])

# np.arange(start, stop) — stop is not included
# Generates [1, 2, 3, ..., 30] for the x-axis
days = np.arange(1, 31)

# Open figure canvas — 10 inches wide, 4 inches tall
plt.figure(figsize=(10, 4))

# Bar chart: one bar per day
# steelblue fill, thin white border between bars for readability
plt.bar(days, rainfall, color='steelblue', edgecolor='white', linewidth=0.4)

# Horizontal reference line at IMD Heavy Rain threshold
# Any bar crossing this line = dam safety alert was triggered that day
plt.axhline(64.5, color='orange', linestyle='--', linewidth=1.5,
            label='IMD Heavy Rain (64.5 mm)')

# Horizontal reference line at the monthly mean
# f-string computes and embeds the mean directly in the label text
plt.axhline(rainfall.mean(), color='red', linestyle=':', linewidth=1.2,
            label=f'Monthly mean ({rainfall.mean():.1f} mm)')

plt.xlabel('Day of June 2024')
plt.ylabel('Rainfall (mm)')
plt.title('Daily Rainfall — Cauvery Basin, June 2024')
plt.legend()
plt.tight_layout()   # prevents axis labels from being clipped at the edges
plt.show()

# Count days at or above Heavy Rain threshold
# rainfall >= 64.5 → [F, F, F, F, ..., T, T, ...]
# .sum() → counts True as 1, False as 0
print(f"Days with Heavy+ rainfall (>= 64.5 mm): {(rainfall >= 64.5).sum()}")

---
## Code Block 3 — Creating Arrays Automatically

### What this code does

Instead of typing every value by hand, NumPy provides functions that generate arrays for you. We use these to create a range of pipe diameters for a design table and a range of slopes for a parametric Manning's discharge study — then compute discharge for all slopes at once.

### Why each step is taken

**`np.arange(150, 650, 50)` for pipe diameters:**
This generates `[150, 200, 250, 300, 350, 400, 450, 500, 550, 600]` — ten standard pipe diameters in mm. Writing these ten values by hand would work, but `arange` is more robust: if you later change the step to 25 mm, you change one number instead of rewriting the whole list. `arange(start, stop, step)` works exactly like Python's `range()` but returns an ndarray.

**`np.linspace(0.0005, 0.005, 10)` for slopes:**
`linspace` generates exactly 10 values equally spaced between 0.0005 and 0.005. Unlike `arange` where you specify the step, here you specify how many values you want. This is ideal when you want a fixed number of evenly distributed sample points across a design range.

**Vectorised Manning's formula — why no loop?**
Manning's formula is `Q = (1/n) × R^(2/3) × S^(1/2) × A`. The only variable changing across the 10 cases is `S` (slope). Because `slopes` is a NumPy array, writing `slopes**0.5` computes the square root of all 10 values simultaneously inside NumPy's C code. The result `Q` is also an array of 10 values — one per slope. This replaces a for loop entirely.

A drainage engineer doing a sensitivity analysis — "how does discharge change as slope varies from 1:2000 to 1:200?" — gets all 10 answers in one expression instead of writing a loop.

### Algorithm

```
1. np.arange(150, 650, 50)
   → generates [150, 200, 250, ..., 600] mm
   — ten standard pipe diameters for a design table

2. np.linspace(0.0005, 0.005, 10)
   → generates 10 slopes equally spaced across the design range

3. Fix pipe parameters: D = 0.3 m, n = 0.013
   Compute:
     A = π × (D/2)²    (cross-sectional area, m²)
     R = D / 4          (hydraulic radius for full circular pipe, m)

4. Vectorised Manning's formula:
     Q = (1/n) × R^(2/3) × slopes^0.5 × A
   slopes^0.5 applies to ALL 10 slope values at once
   Q is an array of 10 discharge values — no loop written

5. Print diameters, slopes, and discharges (rounded for readability)
```

In [ ]:
import numpy as np
import math

# np.arange(start, stop, step)
# Generates values from start up to (not including) stop, stepping by step
# Returns an ndarray — same idea as Python's range() but much more powerful
diameters_mm = np.arange(150, 650, 50)    # [150, 200, 250, ..., 600]
print("Pipe diameters (mm):", diameters_mm)

# np.linspace(start, stop, num)
# Generates exactly num values equally spaced between start and stop
# Both endpoints are included — unlike arange which excludes the stop value
slopes = np.linspace(0.0005, 0.005, 10)
print("Slopes            :", np.round(slopes, 4))

# Manning's discharge for a fixed D=0.3 m concrete pipe (n=0.013)
D = 0.3       # pipe diameter, m
n = 0.013     # Manning's roughness coefficient for concrete
A = math.pi * (D/2)**2   # cross-sectional area = π r²  (m²)
R = D / 4                 # hydraulic radius for a full circular pipe (m)

# Vectorised formula — slopes is an array of 10 values
# slopes**0.5 computes the square root of all 10 elements simultaneously
# The result Q is an array: one discharge value for each slope
Q = (1/n) * R**(2/3) * slopes**0.5 * A

# np.round(array, decimals) rounds every element of the array
print("Discharge (L/s)   :", np.round(Q * 1000, 2))

---
## Code Block 4 — Statistical Analysis of the Annual Maximum Series

### What this code does

We load a 25-year Annual Maximum Series (AMS) from the Cauvery basin and compute the key descriptive statistics used in flood frequency analysis — mean, standard deviation, min, max, median, and quartiles.

### Background — what is an AMS?

The **Annual Maximum Series** is a standard dataset in hydrology:
- For each year of record, take the **single highest peak flow** measured in that year
- A 25-year AMS therefore has 25 values — one per year
- These values are the input to flood frequency analysis (Weibull, Gumbel, Log-Pearson)
- From the AMS, engineers estimate the T-year design flood — for example, the 100-year flood used to size a dam spillway

### Why each step is taken

**`AMS.mean()` — the average annual peak flood:**
This gives the baseline expectation for what the river produces in a typical year. If the mean is 900 m³/s, a minor monsoon might produce 400 m³/s and a strong one 2000 m³/s.

**`AMS.std()` — the standard deviation:**
Measures spread. A high standard deviation means flood peaks vary a lot from year to year — the river is flashy and unpredictable. A low standard deviation means relatively consistent peaks, which makes design easier.

**`AMS.min()` and `AMS.max()`:**
The record low and record high peak flows observed over 25 years. The max is particularly important — it is the largest flood ever recorded and sets a lower bound on any dam spillway capacity.

**`np.median(AMS)` — the median:**
The middle value when the data is sorted. If the median is much lower than the mean, the distribution is right-skewed — a few very large floods are pulling the mean up. The median is a more robust indicator of a typical year.

**`np.percentile(AMS, 25)` and `np.percentile(AMS, 75)`:**
Q25 is the value exceeded in 75% of years (a relatively common flood). Q75 is exceeded in only 25% of years (a rarer, larger flood). The IQR = Q75 − Q25 is a robust measure of spread that is not distorted by the very largest or smallest values.

**Why `np.median()` and `np.percentile()` are functions, not methods:**
These require sorting the data internally before computing, which is a different kind of operation from simple statistics like sum and mean. NumPy exposes them as standalone functions rather than array methods.

### Algorithm

```
1. Load 25-year AMS as a NumPy array
   — one value per year, each is the annual peak flow in m³/s

2. AMS.size       → number of years in the record

3. AMS.mean()     → average annual peak flood
   AMS.std()      → standard deviation (spread)
   AMS.min()      → lowest year on record
   AMS.max()      → highest year on record

4. np.median(AMS)
   — internally sorts the array
   — returns the middle value (or average of two middle values)
   — compare with mean to check for skewness

5. np.percentile(AMS, 25) → Q25 (exceeded in 75% of years)
   np.percentile(AMS, 75) → Q75 (exceeded in 25% of years)
   IQR = Q75 − Q25        → robust measure of spread

6. Print all statistics with labels and units
```

In [ ]:
import numpy as np

# 25-year Annual Maximum Series (m³/s) — Cauvery basin
# Each value is the highest peak flow recorded in that calendar year
AMS = np.array([345.2, 567.8, 892.3, 1234.5, 456.7, 789.0, 1567.8,
                678.9, 345.6, 1890.2, 567.3, 789.4, 2134.6, 456.8,
                678.9, 1234.5, 345.7, 890.2, 1456.7, 567.4, 890.3,
                1123.4, 678.5, 456.9, 1567.2])

print("AMS Flood Statistics — Cauvery Basin")
print("-" * 42)

# .size gives total number of elements (equivalent to len() for 1-D arrays)
print(f"Record length : {AMS.size} years")

# Array methods — operate on all elements, no loop needed
print(f"Mean          : {AMS.mean():.1f} m³/s")
print(f"Std deviation : {AMS.std():.1f} m³/s")
print(f"Minimum       : {AMS.min():.1f} m³/s")
print(f"Maximum       : {AMS.max():.1f} m³/s")

# np.median() is a standalone function — it sorts internally before finding the middle value
print(f"Median        : {np.median(AMS):.1f} m³/s")

# np.percentile(array, p) — p is the percentile between 0 and 100
q25 = np.percentile(AMS, 25)   # Q25 — value exceeded in 75% of years
q75 = np.percentile(AMS, 75)   # Q75 — value exceeded in only 25% of years
print(f"Q25           : {q25:.1f} m³/s")
print(f"Q75           : {q75:.1f} m³/s")

# IQR = Inter-Quartile Range — robust measure of spread
print(f"IQR           : {q75 - q25:.1f} m³/s")

---
## Day 19 Assignment

Apply everything from today's session to the July 2024 streamflow record below.

```python
flows = np.array([234, 267, 312, 890, 1245, 987, 756, 543, 412, 345,
                  289, 245, 212, 198, 220, 265, 310, 456, 678, 890,
                  1123, 987, 765, 543, 421, 345, 289, 245, 212, 198, 210])
```

**Tasks:**
1. Print dtype, shape, total, mean, max, min, std, median
2. Count the number of days where flow exceeds the mean
3. Plot as a bar chart with a mean reference line
4. Find the day number (1-based) of the peak flow

`flows.argmax()` returns the 0-based index of the maximum. Add 1 to convert to a day number.

### ▶ Assignment cell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

flows = np.array([234, 267, 312, 890, 1245, 987, 756, 543, 412, 345,
                  289, 245, 212, 198, 220, 265, 310, 456, 678, 890,
                  1123, 987, 765, 543, 421, 345, 289, 245, 212, 198, 210])

# 1. Statistics
print(f"Mean: {flows.mean():.1f}  Max: {flows.max()}  Min: {flows.min()}")
print(f"Std : {flows.std():.1f}  Median: {np.median(flows):.1f}")

# 2. Days above mean
above_mean = ???
print(f"Days above mean: {above_mean}")

# 3. Plot
days = np.arange(1, len(flows) + 1)
plt.figure(figsize=(10, 4))
plt.bar(days, flows, color='steelblue')
plt.axhline(flows.mean(), color='red', linestyle='--', label='Mean')
plt.xlabel('Day'); plt.ylabel('Flow (m³/s)')
plt.title('July 2024 Streamflow — KRS Station')
plt.legend(); plt.tight_layout(); plt.show()

# 4. Peak day (1-based)
peak_day = ???
print(f"Peak flow on Day: {peak_day}")

---
- [ ] Run all cells from top to bottom
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day19.ipynb`
- [ ] Commit message: `Day 19 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*